In [0]:
/*
  Script: Test Silver
  Description:
    This script validates data quality in the silver layer.
    It checks for nulls, duplicates, invalid formats, inconsistent relationships,
    and ensures all standardized values conform to expected rules across CRM and ERP tables.
*/

CREATE OR REPLACE PROCEDURE test_silver()
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN

  /*
  ================================================================
  Table: crm_cust_info
  ================================================================
  */

  -- Test: No null ids remain
  SELECT *
  FROM silver.crm_cust_info
  WHERE cst_id IS NULL
  ;

  -- Test: Only one history per cst_key + cst_id
  SELECT
    STRUCT(cst_key, cst_id) AS cst_com,
    COUNT(DISTINCT cst_create_date) AS count_date
  FROM silver.crm_cust_info
  GROUP BY cst_com
  ORDER BY count_date DESC
  ;

  -- Test: String columns have no unwanted spaces
  SELECT *
  FROM silver.crm_cust_info
  WHERE
    cst_firstname != TRIM(cst_firstname) OR
    cst_lastname != TRIM(cst_lastname) OR
    cst_marital_status != TRIM(cst_marital_status) OR
    cst_gndr != TRIM(cst_gndr)
  ;

  -- Test: Gender values standardized correctly
  SELECT DISTINCT cst_gndr
  FROM silver.crm_cust_info
  WHERE cst_gndr NOT IN ('Male', 'Female', 'n/a')
  ;

  -- Test: Marital status values standardized correctly
  SELECT DISTINCT cst_marital_status
  FROM silver.crm_cust_info
  WHERE cst_marital_status NOT IN ('Single', 'Married', 'n/a')
  ;

  /*
  ================================================================
  Table: crm_prd_info
  ================================================================
  */

  -- Test: Category ids match source table
  SELECT *
  FROM silver.crm_prd_info
  WHERE REPLACE(SUBSTRING(prd_key, 1, 5), '-', '_') 
        NOT IN (SELECT DISTINCT id FROM silver.erp_px_cat_g1v2)
  ;

  -- Test: Unwanted spaces
  SELECT prd_nm
  FROM silver.crm_prd_info
  WHERE prd_nm != TRIM(prd_nm)
  ;

  -- Test: prd_line within accepted values
  SELECT 
  *
  FROM silver.crm_prd_info
  WHERE prd_line NOT IN ('Mountain', 'Road', 'Other Sales', 'Touring', 'n/a')
  ;

  /*
  ================================================================
  Table: crm_sales_details
  ================================================================
  */

  -- Test: Unwanted spaces
  SELECT *
  FROM silver.crm_sales_details
  WHERE sls_ord_num != TRIM(sls_ord_num)
  ;

  -- Test: Product key match source table
  SELECT *
  FROM silver.crm_sales_details
  WHERE sls_prd_key
        NOT IN (SELECT prd_key FROM silver.crm_prd_info)
  ;

  -- Test: Customer id match source table
  SELECT *
  FROM silver.crm_sales_details
  WHERE sls_cust_id
        NOT IN (SELECT cst_id FROM silver.crm_cust_info)
  ;

  -- Test: Invalid dates
  SELECT *
  FROM silver.crm_sales_details
  WHERE sls_order_dt > sls_ship_dt
    OR sls_order_dt > sls_due_dt
  ;

  -- Test: Invalid ranges
  SELECT *
  FROM silver.crm_sales_details
  WHERE sls_price < 0
  ;

  -- Test: Invalid ranges
  SELECT *
  FROM silver.crm_sales_details
  WHERE sls_quantity < 0
  ;

  -- Test: Invalid derived values
  SELECT
  *
  FROM silver.crm_sales_details
  WHERE sls_sales != sls_quantity * sls_price
  ;

  /*
  ================================================================
  Table: erp_cust_az12
  ================================================================
  */

  -- Test: cid start with a standard prefix
  SELECT
  *
  FROM silver.erp_cust_az12
  WHERE cid NOT LIKE 'AW%'
  ;

  -- Test: bdate is in valid range
  SELECT
  *
  FROM silver.erp_cust_az12
  WHERE bdate < '1924-01-01' OR bdate > GETDATE()
  ;

  -- Test: gen is in valid value set
  SELECT
  *
  FROM silver.erp_cust_az12
  WHERE gen NOT IN ('Male', 'Female', 'n/a')
  ;

  /*
  ================================================================
  Table: erp_loc_a101
  ================================================================
  */

  -- Test: cid match cst_key from other source table
  SELECT
    *
  FROM silver.erp_loc_a101
  WHERE cid NOT IN (
    SELECT DISTINCT cst_key FROM silver.crm_cust_info
  )
  ;

  -- Test: cntry is in valid values
  SELECT *
  FROM silver.erp_loc_a101
  WHERE TRIM(cntry) IS NULL
    OR TRIM(cntry) = ''
  ;

  /*
  ================================================================
  Table: erp_px_cat_g1v2
  ================================================================
  */

  -- Test: id has all categories from crm_prd_info
  SELECT
  *
  FROM silver.crm_prd_info
  WHERE cat_id NOT IN (
    SELECT id FROM silver.erp_px_cat_g1v2
  )
  ;

  -- Test: string columns has no unwanted white space
  SELECT
  *
  FROM silver.erp_px_cat_g1v2
  WHERE
    id != TRIM(id) OR
    subcat != TRIM(subcat) OR
    maintenance != TRIM(maintenance)
  ;

  SELECT 'SUCCESS. All tasks performed successfully.';

END;

-- Example
-- CALL test_silver();